# Set up

In [ ]:
# install libraries
!pip install datasets
!pip install torch torchvision transformers

In [49]:
# Import libraries
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import torch
import torchvision
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer

In [ ]:
# check gpu is available
if torch.cuda.is_available():
  print("GPU is available")
else:
  print("GPU is not available")

GPU is available


In [ ]:
# Import Go-Emotions
emotions_db = load_dataset("mrm8488/goemotions")

README.md:   0%|          | 0.00/7.11k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


goemotions.csv: reconstructing file:   0%|          |  0.00B / 42.7MB            

goemotions.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/211225 [00:00<?, ? examples/s]

In [ ]:
# Inspect dataset
emotions_db.set_format(type="pandas")


                                                text       id  \
0                                    That game hurt.  eew5j0j   
1   >sexuality shouldn’t be a grouping category I...  eemcysk   
2     You do right, if you don't care then fuck 'em!  ed2mah1   
3                                 Man I love reddit.  eeibobj   
4  [NAME] was nowhere near them, he was by the Fa...  eda6yn6   

                author            subreddit    link_id   parent_id  \
0                Brdd9                  nrl  t3_ajis4z  t1_eew18eq   
1          TheGreen888     unpopularopinion  t3_ai4q37   t3_ai4q37   
2             Labalool          confessions  t3_abru74  t1_ed2m7g7   
3        MrsRobertshaw             facepalm  t3_ahulml   t3_ahulml   
4  American_Fascist713  starwarsspeculation  t3_ackt2f  t1_eda65q2   

    created_utc  rater_id  example_very_unclear  admiration  ...  love  \
0  1.548381e+09         1                 False           0  ...     0   
1  1.548084e+09        37               

# Pre processing

In [37]:
# Audio2Face emotions
columns =  [
    'amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral'
]

# Custom dataframe
custom_df = pd.DataFrame(columns = columns)
train_db = emotions_db['train'].to_pandas()

# Emotion columns
custom_df['amazement'] = train_db['amusement'] + train_db['realization'] + train_db['surprise'] + train_db['admiration']
custom_df['anger'] = train_db['anger'] + train_db['annoyance'] + train_db['disapproval']
custom_df['cheekiness'] = train_db['caring'] + train_db['love'] + train_db['desire']
custom_df['disgust'] = train_db['disgust'] + train_db['remorse']
custom_df['fear'] = train_db['fear'] + train_db['nervousness']
custom_df['grief'] = train_db['grief']
custom_df['joy'] = train_db['joy'] + train_db['pride'] + train_db['optimism'] + train_db['gratitude'] + train_db['relief'] + train_db['excitement']
custom_df['out of breath'] = train_db['confusion']
custom_df['pain'] = train_db['disappointment']
custom_df['sadness'] = train_db['sadness'] + train_db['embarrassment']
custom_df['neutral'] = train_db['neutral'] + train_db['approval'] + train_db['curiosity']

# Text column
custom_df['text'] = train_db['text']

# Remove columns with all elements zero
custom_df = custom_df[(custom_df.T != 0).any()]
custom_dataset = Dataset.from_pandas(train_db)

In [34]:
# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Pre processing
def pre_process(examples):
  return tokenizer(examples['text'][0], truncation = True, padding = 'max_length')

# Apply tokenization with batches
distil_dataset = custom_dataset.map(pre_process, batch_size = 10)

Map:   0%|          | 0/211225 [00:00<?, ? examples/s]

# Split into train, test and evaluation data

In [52]:
ds_train_devtest = distil_dataset.train_test_split(test_size=0.2, seed=42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size=0.5, seed=42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'eval': ds_devtest['train'],
    'test': ds_devtest['test']
})


# Fine tuning

In [44]:
# Load DistilBERT model for classification
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=11)

# Setting up training settings
training_args = TrainingArguments(
    output_dir="./results",          # Directory for saving results
    learning_rate=5e-5,              # Initial learning rate
    per_device_train_batch_size=16,  # Batch size per GPU
    num_train_epochs=3,              # Number of epochs
    weight_decay=0.01,               # Regularization
    logging_dir="./logs",            # Directory for logs
    logging_steps=10                 # Log every 10 steps
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
print(ds_splits)
trainer = Trainer(
    model = model,                             # The DistilBERT model
    args = training_args,                      # Training arguments
    train_dataset = ds_splits['train'],      # Training data
    eval_dataset= ds_splits['eval'],          # Validation data
)

# Start training
trainer.train()

# Evaluation

# Deployment